In [1]:
def num(a, b):
    return a+b, a-b

In [2]:
num(5, 4)

(9, 1)

In [5]:
a, b = num(5, 4)

In [9]:
a[0]

9

In [7]:
a = num(5, 4)

In [11]:
!pip  install webdriver_manager


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
_HAS_SELENIUM = False
try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.chrome.service import Service as ChromeService
    from webdriver_manager.chrome import ChromeDriverManager

    _HAS_SELENIUM = True
except ImportError:
    pass

import re

In [13]:
def download_fresh_proxies(proxy_file: str) -> int:
    """
    Download a fresh proxy list from free-proxy-list.net using Selenium.
    Returns the number of proxies downloaded, or 0 on failure.

    Requires: selenium, webdriver-manager (optional dependencies).
    If not installed, logs a warning and returns 0.
    """
    if not _HAS_SELENIUM:
        print("  [proxy-refresh] Selenium not installed -- cannot auto-refresh proxies.")
        print("  [proxy-refresh] Install with: pip install selenium webdriver-manager")
        return 0

    print("  [proxy-refresh] Downloading fresh proxy list...")
    driver = None
    try:
        options = webdriver.ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--log-level=3")
        driver = webdriver.Chrome(
            service=ChromeService(ChromeDriverManager().install()),
            options=options,
        )

        driver.get("https://free-proxy-list.net/en/")
        wait = WebDriverWait(driver, 10)

        button = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'a[title="Get raw list"]'))
        )
        button.click()

        textarea = wait.until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, "textarea.form-control"))
        )
        content = textarea.get_attribute("value")
        proxies = re.findall(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}:\d+", content)

        with open(proxy_file, "w", encoding="utf-8") as f:
            f.write("\n".join(proxies))

        print(f"  [proxy-refresh] OK: Downloaded {len(proxies)} proxies")
        return len(proxies)

    except Exception as e:
        print(f"  [proxy-refresh] FAIL: {e}")
        return 0
    finally:
        if driver:
            driver.quit()

In [16]:
download_fresh_proxies("proxy.txt")

  [proxy-refresh] Downloading fresh proxy list...
  [proxy-refresh] OK: Downloaded 300 proxies


300

In [ ]:
with open("proxy.txt", encoding="utf-8") as f:
    f.read()

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

api = youtube_transcript_api()
url = "https://www.youtube.com/watch?v=OBcvAg0ZSco"
api.fetch(url)

ProxyError: HTTPSConnectionPool(host='www.youtube.com', port=443): Max retries exceeded with url: /watch?v=OBcvAg0ZSco (Caused by ProxyError('Unable to connect to proxy', SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)'))))